In [1]:
import json, os
from bert4torch.models import build_transformer_model
from bert4torch.tokenizers import Tokenizer, load_vocab
from bert4torch.snippets import sequence_padding, seed_everything, ListDataset
from bert4torch.generation import AutoRegressiveDecoder
from bert4torch.callbacks import Callback
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from rouge import Rouge  # pip install rouge
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction # pip install nltk
import jieba

jieba.initialize()

/usr/local/lib/python3.11/site-packages/torch/cuda/__init__.py:56: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Loading model cost 1.500 seconds.
Prefix dict has been built successfully.


In [2]:
class MyDataset(ListDataset):
    @staticmethod
    def load_data(filename):
        D = []
        with open(filename, "r", encoding='utf-8') as reader:
            for line in reader:
                line = line.strip()
                try:
                    text1, text2 = line.split("，")
                    text2 = text2[:-1]
                    D.append((f"{text1[0]}{text2[0]}", line))  # 上下联第一个字 ---> 完整的对联
                except:
                    pass
        return D


In [4]:
def collate_fn(batch):
    """单条样本格式：content：[CLS]文章[SEP]  tgt: [CLS]标题[SEP]
    """
    batch_content_ids, batch_titile_ids = [], []
    for upper, lower in batch:
        token_ids, _ = tokenizer.encode(upper, maxlen=max_c_len)
        batch_content_ids.append(token_ids)

        token_ids, _ = tokenizer.encode(lower, maxlen=max_t_len)
        batch_titile_ids.append(token_ids)

    batch_content_ids = torch.tensor(sequence_padding(batch_content_ids), dtype=torch.long, device=device)
    batch_titile_ids = torch.tensor(sequence_padding(batch_titile_ids), dtype=torch.long, device=device)
    return [[batch_content_ids], [batch_titile_ids[:, :-1]]], batch_titile_ids[:, 1:].flatten()


In [5]:

class CrossEntropyLoss(nn.CrossEntropyLoss):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def forward(self, outputs, y_true):
        y_pred = outputs[-1]
        y_pred = y_pred.reshape(-1, y_pred.shape[-1])
        return super().forward(y_pred, y_true)


In [6]:
class AutoTitle(AutoRegressiveDecoder):
    """seq2seq解码器
    """

    @AutoRegressiveDecoder.wraps(default_rtype='logits')
    def predict(self, inputs, output_ids, states):
        # inputs中包含了[decoder_ids, encoder_hidden_state, encoder_attention_mask]
        res = model.decoder.predict([output_ids] + inputs)
        return res[-1][:, -1, :] if isinstance(res, list) else res[:, -1, :]  # 保留最后一位

    def generate(self, text, topk=1):
        token_ids, _ = tokenizer.encode(text, maxlen=max_c_len)
        token_ids = torch.tensor([token_ids], device=device)
        encoder_output = model.encoder.predict([token_ids])
        output_ids = self.beam_search(encoder_output, topk=topk)[0]  # 基于beam search
        return tokenizer.decode([int(i) for i in output_ids.cpu().numpy()])


In [15]:
def just_show():
    texts = [
        '日月'
    ]
    for text in texts:
        print(text, "  ", autotitle.generate(text))


class Evaluator(Callback):
    """评估与保存
    """

    def __init__(self):
        super(Evaluator, self).__init__()
        self.rouge = Rouge()
        self.smooth = SmoothingFunction().method1
        self.best_bleu = 0.

    def on_epoch_end(self, steps, epoch, logs=None):
        just_show()
        metrics = self.evaluate(valid_dataset.data)  # 评测模型
        metrics_test = self.evaluate(test_dataset.data)  # 评测模型
        if metrics['bleu'] > self.best_bleu:
            self.best_bleu = metrics['bleu']
            model.save_weights('./best_model_v0_b.pt')  # 保存模型
        metrics['best_bleu'] = self.best_bleu
        print('valid_data:', metrics)
        print('test_data:', metrics_test)

    def evaluate(self, data, topk=1):
        total = 0
        rouge_1, rouge_2, rouge_l, bleu = 0, 0, 0, 0
        for upper, lower in tqdm(data):
            total += 1
            title = ' '.join(lower).lower()
            pred_title = ' '.join(autotitle.generate(upper, topk)).lower()
            if pred_title.strip():
                scores = self.rouge.get_scores(hyps=pred_title, refs=title)
                rouge_1 += scores[0]['rouge-1']['f']
                rouge_2 += scores[0]['rouge-2']['f']
                rouge_l += scores[0]['rouge-l']['f']
                bleu += sentence_bleu(
                    references=[title.split(' ')], hypothesis=pred_title.split(' '),
                    smoothing_function=self.smooth
                )
        rouge_1, rouge_2, rouge_l, bleu = rouge_1 / total, rouge_2 / total, rouge_l / total, bleu / total
        return {'rouge-1': rouge_1, 'rouge-2': rouge_2, 'rouge-l': rouge_l, 'bleu': bleu}

# 开始运行

In [16]:
# %% 属性定义

max_c_len = 256
max_t_len = 32
batch_size = 64
epochs = 50
steps_per_epoch = None

pretrain_model = r"/mnt/workspace/models/chinese_t5_pegasus_small"
config_path = os.path.join(pretrain_model, 'bert4torch_config.json')
checkpoint_path = os.path.join(pretrain_model, 'pytorch_model.bin')
dict_path = os.path.join(pretrain_model, 'vocab.txt')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
seed_everything(42)

[INFO] Global seed set to 42


42

In [17]:
tokenizer = Tokenizer(
    dict_path,
    do_lower_case=True,
    pre_tokenize=lambda s: jieba.cut(s, HMM=False)
)


train_dataset = MyDataset("../../datas/poetry_train.txt")

train_dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn
)
valid_dataset = MyDataset("../../datas/poetry_min.txt")
test_dataset = MyDataset("../../datas/poetry_min.txt")

In [18]:
model = build_transformer_model(config_path, checkpoint_path, add_trainer=True).to(device)
model.compile(loss=CrossEntropyLoss(ignore_index=0), optimizer=optim.Adam(model.parameters(), 1e-4))

In [19]:
model

BertBaseModel(
  (encoder): T5_Encoder(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(50000, 512, padding_idx=0)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoderLayer): ModuleList(
      (0-7): 8 x T5Layer(
        (multiHeadAttention): T5Attention(
          (q): Linear(in_features=512, out_features=384, bias=False)
          (k): Linear(in_features=512, out_features=384, bias=False)
          (v): Linear(in_features=512, out_features=384, bias=False)
          (o): Linear(in_features=384, out_features=512, bias=False)
          (relative_positions): T5PositionsEncoding()
          (relative_positions_encoding): Embedding(32, 6)
        )
        (attnLayerNorm): LayerNorm((512,), eps=1e-12, norm_mode=rmsnorm, bias=False)
        (feedForward): T5PositionWiseFeedForward(
          (intermediateDense): Linear(in_features=512, out_features=1024, bias=False)
          (outputDense): Linear(in_features=1024, out_features=512, bias=False)
          (

In [21]:
autotitle = AutoTitle(
    bos_token_id=tokenizer._token_start_id,
    eos_token_id=tokenizer._token_end_id,
    max_new_tokens=max_t_len,
    device=device,
    top_k=2
)

## 训练代码

In [25]:
evaluator = Evaluator()
print(u'生成下联:', autotitle.generate(u'日月'))
model.fit(
    train_dataloader,
    steps_per_epoch=steps_per_epoch,
    epochs=epochs,
    callbacks=[evaluator]
)


生成下联: 月明无归路，月明无归路。
2026-01-28 20:40:54 - Start Training

2026-01-28 20:40:54 - Start  Epoch: 1/50
1127/1127 [==============================] - 2:34 137ms/step - loss: 3.4881 
2026-01-28 20:43:29 - Finish Epoch: 1/50 - loss: 3.5529
日月    日暮江上月，月明江上云。


100%|██████████| 8/8 [00:01<00:00,  4.95it/s]


valid_data: {'rouge-1': 0.3867890815943201, 'rouge-2': 0.10653408599698178, 'rouge-l': 0.36805555062500006, 'bleu': 0.04943584571052436, 'best_bleu': 0.04943584571052436}
test_data: {'rouge-1': 0.3867890815943201, 'rouge-2': 0.10653408599698178, 'rouge-l': 0.36805555062500006, 'bleu': 0.04943584571052436}

2026-01-28 20:43:33 - Start  Epoch: 2/50
  67/1127 [>.............................] - ETA: 0:09<2:26 - loss: 3.3762 

KeyboardInterrupt: 

In [26]:
just_show()

日月    日暮江上月，月明江上云。


## 模型恢复 + 测试

In [27]:
model.load_weights("best_model_v0_b.pt")

In [28]:
for upper in [
    '水火',
    '五四',
    '碧红',
    '金东'
]:
    print(u'生成完整的对联:  ', upper, '  ', autotitle.generate(upper, topk=10))

生成完整的对联:   水火    火开金，火散玉壶杯。
生成完整的对联:   五四    五马一千骑，五马一千骑。
生成完整的对联:   碧红    碧落花初落，红开叶正凋。
生成完整的对联:   金东    金谷人不识，东去有谁知。
